In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Initialize SparkSession with Delta Lake and Iceberg support
spark = SparkSession.builder \
    .appName("Hello Delta and Iceberg Tables") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hadoop") \
    .config("spark.sql.catalog.spark_catalog.warehouse", "file:///home/jovyan/iceberg") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.2.0") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/delta_lake") \
    .getOrCreate()

# Define a sample schema
schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), False)
])

# Create a sample DataFrame
data = [(1, 'Alice'), (2, 'Bob'), (3, 'Charlie')]
df = spark.createDataFrame(data, schema)

# Show the DataFrame
print("Sample DataFrame:")
df.show()

# Save the DataFrame as a Delta table
delta_path = "/home/jovyan/delta_table"
df.write.format("delta").mode("overwrite").save(delta_path)
print(f"Delta table saved at: {delta_path}")

# Register the Delta table in Spark's catalog
spark.sql(f"CREATE TABLE IF NOT EXISTS delta_table USING DELTA LOCATION '{delta_path}'")
print("Delta table registered.")

# Save the DataFrame as an Iceberg table
iceberg_path = "local.db.iceberg_table"
df.write.format("iceberg").mode("overwrite").saveAsTable(iceberg_path)
print(f"Iceberg table saved at: {iceberg_path}")

# Register the Iceberg table in Spark's catalog
spark.sql(f"CREATE TABLE IF NOT EXISTS iceberg_table USING ICEBERG LOCATION '{iceberg_path}'")
print("Iceberg table registered.")

# Verify that the tables are available in the Spark catalog
print("Tables in the Spark catalog:")
spark.sql("SHOW TABLES").show()

# Test querying the Delta table
print("Querying Delta table:")
spark.sql("SELECT * FROM delta_table").show()

# Test querying the Iceberg table
print("Querying Iceberg table:")
spark.sql("SELECT * FROM iceberg_table").show()


:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.4_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-277f1a32-ff05-43e6-a705-b92a1d743fd5;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.iceberg#iceberg-spark-runtime-3.4_2.12;1.3.0 in central
:: resolution report :: resolve 385ms :: artifacts dl 18ms
	:: modules in use:
	io.delta#delta-core_2.12;2.4.0 from central in [default]
	io.delta#delta-storage;2.4.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.4_2.12;1.3.0 from central in [default]
	---------------------------------------------------------------------
	|                  

Sample DataFrame:


+---+-------+
| id|   name|
+---+-------+
|  1|  Alice|
|  2|    Bob|
|  3|Charlie|
+---+-------+



24/08/31 18:23:26 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Delta table saved at: /home/jovyan/delta_table
Delta table registered.


24/08/31 18:23:42 WARN HadoopTableOperations: Error reading version hint file /home/jovyan/iceberg_warehouse/db/iceberg_table/metadata/version-hint.text
java.io.FileNotFoundException: File /home/jovyan/iceberg_warehouse/db/iceberg_table/metadata/version-hint.text does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:779)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1100)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:769)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hadoop.fs.ChecksumFileSystem$ChecksumFSInputChecker.<init>(ChecksumFileSystem.java:160)
	at org.apache.hadoop.fs.ChecksumFileSystem.open(ChecksumFileSystem.java:372)
	at org.apache.hadoop.fs.FileSystem.open(FileSystem.java:976)
	at org.apache.iceberg.hadoop.HadoopTableOperations.findVersion(HadoopTableOperations.java:318)
	at org.apache.

Iceberg table saved at: local.db.iceberg_table
Iceberg table registered.
Tables in the Spark catalog:
+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|  default|  delta_table|      false|
|  default|iceberg_table|      false|
+---------+-------------+-----------+

Querying Delta table:


+---+-------+
| id|   name|
+---+-------+
|  3|Charlie|
|  1|  Alice|
|  2|    Bob|
+---+-------+

Querying Iceberg table:


Py4JJavaError: An error occurred while calling o42.sql.
: java.util.concurrent.ExecutionException: org.apache.spark.sql.AnalysisException: ICEBERG is not a valid Spark SQL Data Source.
	at org.sparkproject.guava.util.concurrent.AbstractFuture$Sync.getValue(AbstractFuture.java:306)
	at org.sparkproject.guava.util.concurrent.AbstractFuture$Sync.get(AbstractFuture.java:293)
	at org.sparkproject.guava.util.concurrent.AbstractFuture.get(AbstractFuture.java:116)
	at org.sparkproject.guava.util.concurrent.Uninterruptibles.getUninterruptibly(Uninterruptibles.java:135)
	at org.sparkproject.guava.cache.LocalCache$Segment.getAndRecordStats(LocalCache.java:2410)
	at org.sparkproject.guava.cache.LocalCache$Segment.loadSync(LocalCache.java:2380)
	at org.sparkproject.guava.cache.LocalCache$Segment.lockedGetOrLoad(LocalCache.java:2342)
	at org.sparkproject.guava.cache.LocalCache$Segment.get(LocalCache.java:2257)
	at org.sparkproject.guava.cache.LocalCache.get(LocalCache.java:4000)
	at org.sparkproject

In [ ]:

# Stop the Spark session
spark.stop()


In [2]:
# Verify that the tables are available in the Spark catalog
spark.sql("SHOW TABLES").show()


NameError: name 'spark' is not defined